# 01 — Morphisms as the app: ingest → materialize → [UM]

This notebook **is the application**. Each code cell is a step of the app,
and the [UM] (`ewm-app::StateMachine`) runs them against the shared
`StateCache`.

Progression (slow, one morphism at a time):

1. **ingest** — the default padded n-gram morphism: SHA1 of the new HLLSet,
   three pointers per token, LUTs + hllsetLUT side effects;
2. **materialize** — LUT-first, ordered by default, `no_order` option;
3. **the [UM]** — the stateless driver over the shared state, cell by cell.

In [2]:
:dep ewm-app = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ewm-state-machine/crates/ewm-app" }
:dep ewm-git = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ewm-state-machine/crates/ewm-git" }
:dep context-tree = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ewm-state-machine/crates/context-tree" }
:dep hllset-morphisms = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ewm-state-machine/crates/hllset-morphisms" }
:dep hllset-contracts = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ewm-state-machine/crates/hllset-contracts" }
:dep hllset-core = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ewm-state-machine/crates/hllset-core" }

In [3]:
use ewm_app::{StateCache, StateMachine};
use hllset_contracts::BitAddress;
use hllset_morphisms::{
    ingest, materialize, materialize_no_order, CHANNELS, CHANNEL_SEEDS, PAD,
};

println!("notebook app loaded: [UM] + morphisms");

notebook app loaded: [UM] + morphisms


---
## Morphism 1 — ingest

The default case: an ordered token collection in, the **SHA1 of the new
HLLSet** out. The side effects preserve the work — token LUTs and the
hllsetLUT with the three channel HLLSet keys.

In [4]:
let tokens: Vec<Vec<u8>> = ["the", "cat", "sat"]
    .iter()
    .map(|t| t.as_bytes().to_vec())
    .collect();

let ing = ingest(&tokens);
println!("default return — SHA1 of the new HLLSet:");
println!("  projection key: {}", ing.key);
for (ch, key) in ing.keys.iter().enumerate() {
    println!("  G{} key: {}", ch + 1, key);
}
println!(
    "hllsetLUT: {} entries, TH = {:?}",
    ing.hllset_lut.len(),
    ing.keys.iter().map(|k| ing.hllset_lut.th(k)).collect::<Vec<_>>()
);

default return — SHA1 of the new HLLSet:


  projection key: h:a421491d50bfa3742c15672667a0bc6d9a3ed102


  G1 key: h:c1b894b666264b47d78bc061db7e63f17d746207


  G2 key: h:1bbb1d9658640931592121398acae8627f8f03f0


  G3 key: h:b7661bd623bca71dca2464c72d43fd66296706d8


hllsetLUT: 3 entries, TH = [1, 1, 1]


Every real token has **three hashes pointing at it** — its 1-gram, 2-gram
and 3-gram (each channel uses its own seed). Let's check the three pointers
of `"cat"` inside `[the, cat, sat]`.

In [5]:
fn join_gram(parts: &[&[u8]]) -> Vec<u8> {
    if parts.len() == 1 {
        return parts[0].to_vec();
    }
    let mut out = Vec::new();
    for (i, p) in parts.iter().enumerate() {
        if i > 0 {
            out.push(0u8);
        }
        out.extend_from_slice(p);
    }
    out
}

let ing = ingest(["the", "cat", "sat"]);
for ch in 0..CHANNELS {
    let seed = CHANNEL_SEEDS[ch];
    let gram: Vec<u8> = match ch {
        0 => join_gram(&[b"cat"]),
        1 => join_gram(&[b"cat", b"sat"]),
        _ => join_gram(&[b"cat", b"sat", PAD]),
    };
    let addr = BitAddress::of_token_seeded(&gram, seed);
    let atom_set = ing.sketches[ch].has_bit(addr.reg(), addr.tz());
    let fiber_has_cat = ing.luts[ch].fiber(addr.bit()).contains(&b"cat".to_vec());
    println!(
        "channel {}: atom of {:?} set = {}, LUT fiber holds cat = {}",
        ch + 1,
        String::from_utf8_lossy(&gram).replace('\0', "|"),
        atom_set,
        fiber_has_cat
    );
}

channel 1: atom of "cat" set = true, LUT fiber holds cat = true


channel 2: atom of "cat|sat" set = true, LUT fiber holds cat = true


channel 3: atom of "cat|sat|<PAD>" set = true, LUT fiber holds cat = true


()

---
## Morphism 2 — materialize

LUT-first over the three channels; **ordered by default**, `no_order`
returns the plain set.

In [6]:
let tokens: Vec<Vec<u8>> = ["the", "cat", "sat", "on", "the", "mat"]
    .iter()
    .map(|t| t.as_bytes().to_vec())
    .collect();
let ing = ingest(&tokens);

let ordered = materialize(&ing);
println!("ordered restore == original: {}", ordered == tokens);

let set = materialize_no_order(&ing);
println!("no_order set: {:?}", set);

ordered restore == original: true


no_order set: {[99, 97, 116], [109, 97, 116], [111, 110], [115, 97, 116], [116, 104, 101]}


---
## The [UM] — this notebook is the app

The [UM] (`StateMachine`) is stateless: it owns only the store handle.
S(t) and H(t-1) live in the shared `StateCache`. Each cell below is one
turn of the app.

In [7]:
let mut um = StateMachine::new(ewm_git::MemoryStore::default());
let mut cache = StateCache::empty();

let turn1 = [10u32, 20, 30];
let out1 = um.run_turn(&mut cache, &turn1).expect("turn 1");
println!("commit: {}", out1.commit.as_ref().map(|c| c.to_string()).unwrap_or_else(|| "-".into()));
println!("tip:    {}", out1.head.as_ref().map(|h| h.to_string()).unwrap_or_else(|| "-".into()));
println!("S(t) leaves: {}", out1.tree.leaves().len());
println!("full_image: {:?}", out1.full_image);
{
    let v = out1.commit_view.as_ref().expect("root view");
    println!("D={} R={} N={}", v.departed.popcount(), v.retained.popcount(), v.new.popcount());
}

commit: d2ee26ac


tip:    d2ee26ac


S(t) leaves: 1


full_image: [[116, 105, 100, 49, 48], [116, 105, 100, 50, 48], [116, 105, 100, 51, 48]]


D=0 R=0 N=3


()

In [8]:
let turn2 = [20u32, 30, 40];
let out2 = um.run_turn(&mut cache, &turn2).expect("turn 2");
println!("commit: {}", out2.commit.as_ref().map(|c| c.to_string()).unwrap_or_else(|| "-".into()));
{
    let v = out2.commit_view.as_ref().expect("head view");
    println!("D={} R={} N={}", v.departed.popcount(), v.retained.popcount(), v.new.popcount());
}
println!(
    "tree D/R/N: added={:?} retained={:?} removed={:?}",
    out2.diff.added, out2.diff.retained, out2.diff.removed
);

commit: 70dfa612


D=0 R=3 N=1


tree D/R/N: added=["h:f97ac4a7b938f40d31bf300c497cd7bc11d8377b"] retained=["h:480a4fd737c5f205d41cc85bf0c973751a88e68e"] removed=[]


---
## Recovery — pop, not rebuild

Crash: drop the [UM] **and** its cache. The persistent store is untouched; a
fresh [UM] and a restored cache resume from the tip.

In [9]:
let dir = std::env::temp_dir().join("um-notebook-recovery");
let _ = std::fs::remove_dir_all(&dir);

// First [UM] commits one turn, then "crashes" (goes out of scope).
{
    let mut um = StateMachine::new(ewm_git::LooseStore::new(&dir));
    let mut cache = StateCache::empty();
    um.run_turn(&mut cache, &[1u32, 2, 3]).expect("turn");
    println!("tip after first [UM]: {:?}", um.head().map(|h| h.to_string()));
}

// Fresh [UM] over the same store; fresh cache restored from the tip.
let mut um2 = StateMachine::open(ewm_git::LooseStore::new(&dir));
let mut cache2 = StateCache::restore(um2.repo());
println!("restored tip:        {:?}", um2.head().map(|h| h.to_string()));
println!("restored cache.tip:  {:?}", cache2.tip.as_ref().map(|h| h.to_string()));
println!("restored leaves:     {}", cache2.tree().leaves().len());

let resumed = um2.run_turn(&mut cache2, &[3u32, 4]).expect("resume");
println!("resumed commit:      {:?}", resumed.commit.as_ref().map(|c| c.to_string()));
let _ = std::fs::remove_dir_all(&dir);

tip after first [UM]: Some("420b333b")


restored tip:        Some("420b333b")


restored cache.tip:  Some("420b333b")


restored leaves:     1


resumed commit:      Some("6b0341eb")


---
## Summary

- **ingest** returns the SHA1 of the new HLLSet and preserves the work in
  LUTs + hllsetLUT;
- **materialize** restores the ordered token collection (or the plain set
  with `no_order`);
- the **[UM]** runs this notebook cell by cell as a stateless driver, with
  S(t) and H(t-1) outside it in the shareable `StateCache`;
- recovery is a fresh [UM] + `StateCache::restore` — a read of the tip, not
  a replay.